# Deepfake Detection — Colab Training

Train an EfficientNet-B4 deepfake detector on a free Colab T4 GPU.
Runs ~30–45 min for 10 epochs (vs 2–3 hrs on M2 Pro). No load on your laptop.

**Before running this notebook:**

1. **Enable GPU runtime:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save*
2. **Push your project to GitHub** (so this notebook can clone it). From your Mac terminal:
   ```bash
   cd ~/Downloads/deepfake-detection
   git init && git add . && git commit -m "Initial deepfake detector"
   git branch -M main
   git remote add origin https://github.com/Krishna-IITB/deepfake-detection.git
   git push -u origin main
   ```
   Then set the URL in the cell below.
3. **Have `kaggle.json` ready** to upload (the same one from `~/.kaggle/kaggle.json` on your Mac).

Then just *Runtime → Run all* and watch.

## 1. Verify GPU is attached

In [ ]:
!nvidia-smi

Expected: a table showing **Tesla T4** with ~15 GB memory. If you see `command not found`, GPU isn't enabled — go to *Runtime → Change runtime type* and pick T4.

## 2. Install dependencies

In [ ]:
!pip install -q albumentations facenet-pytorch tensorboard
!pip install -q kaggle

## 3. Clone your project from GitHub

Set your repo URL below.

In [ ]:
GITHUB_REPO = "https://github.com/Krishna-IITB/deepfake-detection.git"  # ← change to your repo

import os
!rm -rf deepfake-detection
!git clone {GITHUB_REPO}
%cd deepfake-detection
!ls

Expected: the `src/`, `scripts/`, `app/`, `README.md`, etc. listed.

## 4. Mount Google Drive

Checkpoints and reports get saved to your Drive so they survive when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/deepfake-detection'
!mkdir -p "{DRIVE_DIR}"
print(f"Drive output dir: {DRIVE_DIR}")

## 5. Set up Kaggle API

When you run the next cell, a file picker pops up. Upload your `kaggle.json` from your Mac (it's at `~/.kaggle/kaggle.json` — copy it to your Mac's Desktop first, then upload it here).

In [ ]:
from google.colab import files
import os

print("Upload kaggle.json from your Mac:")
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# verify
!kaggle datasets list -s deepfake | head -3

Expected: 3 rows of Kaggle dataset listings. If you see `401 Unauthorized`, the kaggle.json wasn't valid — re-upload.

## 6. Download the dataset (~3–5 min on Colab)

The 140k Real and Fake Faces dataset, ~4 GB. Already split into train/valid/test, already cropped to faces.

In [ ]:
!mkdir -p data
%cd data
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip
!rm 140k-real-and-fake-faces.zip
%cd ..
!ls data/real_vs_fake/real-vs-fake/

Expected: `train  valid  test`

## 7. Wire dataset paths to what the pipeline expects

Dataset uses `valid/`, pipeline expects `val/` — symlinks fix it.

In [ ]:
!mkdir -p data/frames
!ln -sfn "$(pwd)/data/real_vs_fake/real-vs-fake/train" data/frames/train
!ln -sfn "$(pwd)/data/real_vs_fake/real-vs-fake/valid" data/frames/val
!ln -sfn "$(pwd)/data/real_vs_fake/real-vs-fake/test"  data/frames/test
!ls data/frames/train

Expected: `fake  real`

## 8. (Optional) Quick smoke test — B0, 2 epochs (~5 min)

Confirms the pipeline runs end to end on the T4 before kicking off the real run.
Skip this cell if you want to go straight to the full training in step 9.

In [ ]:
!python -m src.train \
    --train-dir data/frames/train \
    --val-dir   data/frames/val \
    --backbone  efficientnet_b0 \
    --img-size 224 \
    --epochs 2 \
    --batch-size 64 \
    --num-workers 2 \
    --out-dir checkpoints_smoke

Expected: `Device: cuda`, val AUC > 0.95 by epoch 2.

## 9. Real training run — EfficientNet-B4, 10 epochs (~30–45 min on T4)

In [ ]:
!python -m src.train \
    --train-dir data/frames/train \
    --val-dir   data/frames/val \
    --backbone  efficientnet_b4 \
    --img-size 380 \
    --epochs 10 \
    --batch-size 32 \
    --num-workers 2

Saves `checkpoints/best.pt` whenever val AUC improves. TensorBoard logs in `checkpoints/tb/`.

## 10. Evaluate on test set

In [ ]:
!python -m src.evaluate \
    --ckpt checkpoints/best.pt \
    --data-dir data/frames/test \
    --out-dir reports

Prints accuracy / AUC / EER / F1, writes `reports/metrics.json`, `reports/roc.png`, `reports/confusion_matrix.png`.

## 11. Save results to Google Drive

So you can grab the checkpoint + metrics + plots later from your Mac via the Drive web UI.

In [ ]:
!cp checkpoints/best.pt "{DRIVE_DIR}/"
!cp -r reports "{DRIVE_DIR}/"
!cp -r checkpoints/tb "{DRIVE_DIR}/tensorboard_logs"

print(f"Saved to {DRIVE_DIR}:")
!ls -lh "{DRIVE_DIR}/"
!ls -lh "{DRIVE_DIR}/reports/" 

## 12. Inspect results inline

Quick view of the metrics + ROC curve + confusion matrix without leaving the notebook.

In [ ]:
import json
from IPython.display import Image, display

with open('reports/metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

display(Image('reports/roc.png'))
display(Image('reports/confusion_matrix.png'))

## Done

Now back on your Mac:

1. **Pull the checkpoint and reports from Drive:**
   ```bash
   # download via Drive web UI, or use rclone/gdrive CLI
   ```
2. **Update README "Results" table** with the AUC / accuracy / EER you just got.
3. **Commit the screenshots and metrics:**
   ```bash
   git add reports/ assets/
   git commit -m "Add training results: val AUC X.XX"
   git push
   ```
4. **Pin the repo on your GitHub profile** so it shows up first for recruiters.

That's it — your Mac stayed cold, you got real numbers, and the notebook itself is shareable as a "click to reproduce" demo for interviews.